# CLI Select Command

> `kreview select` — Feature scoring and hybrid-union selection.
>
> Reads `*_matrix.parquet` files from `kreview extract`, scores features,
> and applies hybrid-union selection (top N% AUC ∪ top N% MI).

In [ ]:
#| default_exp cli_select

In [ ]:
#| export
"""CLI command: ``kreview select`` — Feature scoring and selection.

Reads ``*_matrix.parquet`` files (from ``kreview extract``), scores every
numeric feature by univariate AUC and mutual information, then applies
feature selection (mRMR by default, or hybrid-union) with a variance guard.

Outputs per evaluator:

- ``{name}_matrix.parquet`` — selected-feature matrix (metadata + top features)
- ``{name}_eval_stats.parquet`` — per-feature scores for ALL features
- ``{name}_selection_qc.json`` — selection audit trail

Registered in the main CLI via::

    from kreview.cli_select import select
    app.command()(select)
"""

import json
import time
from pathlib import Path

import structlog
import pandas as pd
import typer

log = structlog.get_logger()


def select(
    matrices_dir: Path = typer.Option(
        ...,
        "--matrices-dir",
        help="Directory with *_matrix.parquet files from kreview extract",
    ),
    top_percentile: float = typer.Option(
        50.0,
        "--top-percentile",
        help="Top X%% of features to select (controls K for mRMR, or per-metric cutoff for hybrid_union)",
    ),
    strategy: str = typer.Option(
        "mrmr",
        "--strategy",
        help="Feature selection strategy: mrmr (default, redundancy-aware) or hybrid_union (AUC∪MI)",
    ),
    cv_folds: int = typer.Option(
        5, "--cv-folds", help="Cross-validation folds for univariate AUC scoring"
    ),
    impute_strategy: str = typer.Option(
        "median",
        "--impute-strategy",
        help="Imputation strategy for variance check: median, mean, zero",
    ),
    output: Path = typer.Option(
        "output/",
        "--output",
        help="Output directory for selected matrices (ignored when --overwrite is set)",
    ),
    overwrite: bool = typer.Option(
        False,
        "--overwrite",
        help="Overwrite original matrices in --matrices-dir instead of writing to --output",
    ),
    compute_univariate_auc: bool = typer.Option(
        True,
        "--compute-univariate-auc/--no-compute-univariate-auc",
        help="Compute per-feature univariate AUC (disable for MI-only selection)",
    ),
    seed: int = typer.Option(
        42, "--seed", help="Random seed for reproducibility.",
    ),
):
    """Score features and apply feature selection to extracted matrices.

    Reads all ``*_matrix.parquet`` files from ``--matrices-dir``, computes
    feature scores (univariate AUC + mutual information), and selects the
    top features using mRMR (default, redundancy-aware) or hybrid union
    (top N%% by AUC ∪ top N%% by MI).

    Writes selected matrices, eval stats, and QC metadata to ``--output``
    (or overwrites originals when ``--overwrite`` is set).
    """
    from kreview.selection import score_features, select_features
    from kreview.reproducibility import seed_everything

    seed_everything(seed)

    print("=== kreview select ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --matrices-dir          : {matrices_dir}", flush=True)
    print(f"  --top-percentile        : {top_percentile}", flush=True)
    print(f"  --strategy              : {strategy}", flush=True)
    print(f"  --cv-folds              : {cv_folds}", flush=True)
    print(f"  --impute-strategy       : {impute_strategy}", flush=True)
    print(f"  --output                : {output}", flush=True)
    print(f"  --overwrite             : {overwrite}", flush=True)
    print(f"  --compute-univariate-auc: {compute_univariate_auc}", flush=True)
    print("", flush=True)

    log.info(
        "select_start",
        matrices_dir=str(matrices_dir),
        strategy=strategy,
        top_percentile=top_percentile,
        cv_folds=cv_folds,
        impute_strategy=impute_strategy,
        compute_univariate_auc=compute_univariate_auc,
        output=str(output),
        overwrite=overwrite,
    )

    t0 = time.time()

    # ── Validate inputs ──
    if not matrices_dir.exists():
        print(f"ERROR: Matrices directory not found: {matrices_dir}", flush=True)
        raise typer.Exit(code=1)

    # Discover matrix files (exclude super_matrix and scoreboard files)

    matrix_files = sorted(matrices_dir.glob("*_matrix.parquet"))
    matrix_files = [
        f
        for f in matrix_files
        if not f.name.startswith("super_matrix")
        and not f.name.startswith("scoreboard_")
    ]

    if not matrix_files:
        print(
            f"ERROR: No *_matrix.parquet files found in {matrices_dir}",
            flush=True,
        )
        raise typer.Exit(code=1)

    print(f"Found {len(matrix_files)} matrix files:", flush=True)
    for mf in matrix_files:
        print(f"  - {mf.name}", flush=True)
    print("", flush=True)

    # ── Determine output directory ──
    out_dir = matrices_dir if overwrite else Path(output)
    out_dir.mkdir(parents=True, exist_ok=True)

    if overwrite:
        print("Mode: OVERWRITE (writing to --matrices-dir)", flush=True)
    else:
        print(f"Mode: SEPARATE (writing to {out_dir})", flush=True)
    print("", flush=True)

    # ── Process each evaluator matrix ──
    n_processed = 0
    n_skipped = 0

    for mf in matrix_files:
        eval_name = mf.name.replace("_matrix.parquet", "")
        print(f"{'='*60}", flush=True)
        print(f"Processing evaluator: {eval_name}", flush=True)

        t_eval = time.time()

        try:
            df = pd.read_parquet(mf)
        except Exception as exc:
            print(f"  ERROR: Failed to read {mf.name}: {exc}", flush=True)
            log.error("select_read_failed", evaluator=eval_name, error=str(exc))
            n_skipped += 1
            continue

        print(f"  Loaded: {df.shape[0]} samples × {df.shape[1]} columns", flush=True)

        # ── Score features ──
        try:
            eval_stats = score_features(
                df,
                cv_folds=cv_folds,
                compute_auc=compute_univariate_auc,
                random_state=seed,
            )
        except ValueError as exc:
            print(f"  WARNING: Scoring failed for {eval_name}: {exc}", flush=True)
            log.warning(
                "select_scoring_failed",
                evaluator=eval_name,
                error=str(exc),
            )
            n_skipped += 1
            continue

        print(
            f"  Scored: {len(eval_stats)} features "
            f"(AUC>0.55: "
            f"{int((eval_stats['univariate_auc'] > 0.55).sum()) if 'univariate_auc' in eval_stats.columns else 'N/A'}, "
            f"MI>0: {int((eval_stats['mutual_info'] > 0.0).sum())})",
            flush=True,
        )

        # Save eval stats (always — contains scores for ALL features)
        stats_out = out_dir / f"{eval_name}_eval_stats.parquet"
        eval_stats.to_parquet(stats_out, index=False)
        print(f"  Eval stats → {stats_out.name}", flush=True)

        # ── Select features ──
        try:
            selected_df, selection_qc = select_features(
                df,
                eval_stats,
                top_percentile=top_percentile,
                impute_strategy=impute_strategy,
                strategy=strategy,
            )
        except ValueError as exc:
            print(f"  WARNING: Selection failed for {eval_name}: {exc}", flush=True)
            log.warning(
                "select_selection_failed",
                evaluator=eval_name,
                error=str(exc),
            )
            n_skipped += 1
            continue

        # Save selected matrix
        matrix_out = out_dir / f"{eval_name}_matrix.parquet"
        selected_df.to_parquet(matrix_out, index=False)

        # Save selection QC
        qc_out = out_dir / f"{eval_name}_selection_qc.json"
        with open(qc_out, "w") as f:
            json.dump(selection_qc, f, indent=2, default=str)

        elapsed = time.time() - t_eval
        if selection_qc.get("method") == "mrmr":
            print(
                f"  Selected: {selection_qc['n_after_variance_guard']}"
                f"/{selection_qc['total_input_features']} features "
                f"(mRMR, variance-dropped={selection_qc['n_variance_dropped']}) "
                f"in {elapsed:.1f}s",
                flush=True,
            )
        else:
            print(
                f"  Selected: {selection_qc['n_after_variance_guard']}"
                f"/{selection_qc['total_input_features']} features "
                f"(AUC∩MI={selection_qc.get('n_overlap_both', 0)}, "
                f"AUC-only={selection_qc.get('n_auc_only', 0)}, "
                f"MI-only={selection_qc.get('n_mi_only', 0)}) "
                f"in {elapsed:.1f}s",
                flush=True,
            )
        print(f"  Matrix → {matrix_out.name}", flush=True)
        print(f"  QC → {qc_out.name}", flush=True)

        log.info(
            "select_evaluator_complete",
            evaluator=eval_name,
            **selection_qc,
            elapsed_sec=round(elapsed, 1),
        )
        n_processed += 1

    # ── Summary ──
    total_elapsed = time.time() - t0
    print(f"\n{'='*60}", flush=True)
    print(
        f"=== Select complete: {n_processed}/{n_processed + n_skipped} "
        f"evaluators in {total_elapsed:.1f}s ===",
        flush=True,
    )

    if n_skipped > 0:
        print(f"  WARNING: {n_skipped} evaluators skipped (see logs)", flush=True)

    log.info(
        "select_complete",
        n_processed=n_processed,
        n_skipped=n_skipped,
        output_dir=str(out_dir),
        elapsed_sec=round(total_elapsed, 1),
    )

    if n_processed == 0:
        print("ERROR: No evaluators were processed successfully", flush=True)
        raise typer.Exit(code=1)